In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gustjs21@"))

def run_query(tx, query):
    return tx.run(query).data()

# 중심성 분석 알고리즘
algorithms = {
    "degree": "gds.degree.write",
    "betweenness": "gds.betweenness.write",
    "closeness": "gds.closeness.write",
    "eigenvector": "gds.eigenvector.write",
    "pagerank": "gds.pageRank.write"
}

with driver.session(database="sicpama") as session:
    # 모든 DINED_WITH_* 관계 불러오기
    rels = session.execute_read(run_query, """
        CALL db.relationshipTypes() YIELD relationshipType
        WHERE relationshipType STARTS WITH 'DINED_WITH_'
        RETURN relationshipType
    """)
    
    for rel_obj in rels:
        rel_type = rel_obj['relationshipType']
        graph_name = f"graph_{rel_type}"

        print(f"\n🔄 {rel_type} 처리 시작...")

        # 기존 그래프가 있다면 삭제
        try:
            drop_query = f"""
            CALL gds.graph.exists('{graph_name}') YIELD exists
            WITH exists WHERE exists
            CALL gds.graph.drop('{graph_name}') YIELD graphName
            RETURN graphName
            """
            session.execute_write(run_query, drop_query)
            print(f"🧹 이전 그래프 삭제: {graph_name}")
        except:
            pass

        # projection 수행
        try:
            projection_query = f"""
            CALL gds.graph.project.cypher(
              '{graph_name}',
              'MATCH (c:Customer)-[:{rel_type}]-() RETURN id(c) AS id',
              'MATCH (c1:Customer)-[r:{rel_type}]->(c2:Customer)
               RETURN id(c1) AS source, id(c2) AS target, r.times AS times'
            )
            """
            session.execute_write(run_query, projection_query)
            print(f"✅ Projection 완료: {graph_name}")
        except Exception as e:
            print(f"❌ Projection 실패 for {rel_type}: {e}")
            continue  # 다음 store로 넘어감

        # 중심성 분석 수행
        for algo, func in algorithms.items():
            prop_name = f"{algo}_{rel_type}"
            print(f"→ {algo} 계산 중...")

            try:
                algo_query = f"""
                CALL {func}('{graph_name}', {{
                  writeProperty: '{prop_name}'
                }})
                """
                session.execute_write(run_query, algo_query)
                print(f"✅ {algo} 완료 → {prop_name}")
            except Exception as e:
                print(f"❌ {algo} 실패: {e}")


In [ ]:
from neo4j import GraphDatabase
import pandas as pd
from IPython.display import display

# 연결 설정
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gustjs21@"))

def run_query(tx, query):
    return tx.run(query).data()

# 1. 모든 관계 목록 가져오기
get_relationships = """
CALL db.relationshipTypes() YIELD relationshipType
WHERE relationshipType STARTS WITH 'DINED_WITH_'
RETURN relationshipType
"""

with driver.session(database="sicpama") as session:
    rels = session.execute_read(run_query, get_relationships)

# 2. 각 관계 순회하며 customer 초대 횟수 집계
store_data = []

for rel in rels:
    rel_type = rel["relationshipType"]
    store_id = int(rel_type.replace("DINED_WITH_", ""))

    query = f"""
    MATCH (c:Customer)-[r:`{rel_type}`]-(other:Customer)
    RETURN '{store_id}' AS store, c.id AS customerId, sum(COALESCE(r.times, 1)) AS invited_count
    """

    with driver.session(database="sicpama") as session:
        result = session.execute_read(run_query, query)
        store_data.extend(result)

# 3. DataFrame으로 변환
pd.set_option('display.max_rows', None) 
df_invited = pd.DataFrame(store_data)
display(df_invited)


In [ ]:
from neo4j import GraphDatabase
import pandas as pd
from IPython.display import display

# Neo4j 연결 설정
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gustjs21@"))

def run_query(tx, query):
    return tx.run(query).data()

# 1. 모든 관계 목록 가져오기
get_relationships = """
CALL db.relationshipTypes() YIELD relationshipType
WHERE relationshipType STARTS WITH 'DINED_WITH_'
RETURN relationshipType
"""

with driver.session(database="sicpama") as session:
    rels = session.execute_read(run_query, get_relationships)

# 2. 각 관계 순회하며 customer 초대 횟수 집계
store_data = []

for rel in rels:
    rel_type = rel["relationshipType"]
    store_id = int(rel_type.replace("DINED_WITH_", ""))

    query = f"""
    MATCH (c:Customer)-[r:`{rel_type}`]->(:Customer)
    RETURN '{store_id}' AS store,
           COALESCE(c.id, toString(ID(c))) AS customerId,
           sum(COALESCE(r.times, 1)) AS invited_count
    """

    with driver.session(database="sicpama") as session:
        result = session.execute_read(run_query, query)
        store_data.extend(result)

# 3. DataFrame으로 변환 및 출력
pd.set_option('display.max_rows', None)
df_invited = pd.DataFrame(store_data)
display(df_invited)


In [ ]:
# store=2인 항목만 필터링
store_2_data = [row for row in store_data if row['store'] == '2']
print("store=2인 데이터 개수:", len(store_2_data))
print(store_2_data[:3])  # 일부만 출력


In [ ]:
print([rel['relationshipType'] for rel in rels])


In [ ]:
from neo4j import GraphDatabase
import pandas as pd
from IPython.display import display

# Neo4j 연결 설정
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "gustjs21@"))

# 쿼리 실행 함수
def run_query(tx, query):
    return tx.run(query).data()

# 1. 모든 DINED_WITH_* 관계 목록 가져오기
get_relationships = """
CALL db.relationshipTypes() YIELD relationshipType
WHERE relationshipType STARTS WITH 'DINED_WITH_'
RETURN relationshipType
"""

with driver.session(database="sicpama") as session:
    rels = session.execute_read(run_query, get_relationships)

# 2. 각 관계 순회하며 customer 초대 횟수 집계
store_data = []

for rel in rels:
    rel_type = rel["relationshipType"]
    store_id = int(rel_type.replace("DINED_WITH_", ""))

    query = f"""
    MATCH (c:Customer)-[r:`{rel_type}`]->(:Customer)
    RETURN '{store_id}' AS store,
           COALESCE(c.id, toString(ID(c))) AS customerId,
           sum(COALESCE(r.times, 1)) AS invited_count
    """

    with driver.session(database="sicpama") as session:
        result = session.execute_read(run_query, query)

    print(f"{rel_type} → {len(result)} rows")
    store_data.extend(result)

# 3. DataFrame으로 변환 및 null 제거
df_invited = pd.DataFrame(store_data)
df_invited = df_invited[df_invited['invited_count'].notnull()]

# 4. 결과 출력
pd.set_option('display.max_rows', None)
display(df_invited)

# (선택) store=2만 필터링
store_2_data = df_invited[df_invited['store'] == '2']
print("✅ store=2인 데이터 개수:", len(store_2_data))
display(store_2_data.head())
